In [ ]:
import pandas as pd
import numpy as np
import joblib
from scipy.spatial import cKDTree, distance
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

In [ ]:
# Multi-scale spatial basis function generation
def generate_knots(coords, num_nodes_per_dim=5):
    """
    Generate a regular grid of knots within the coordinate range.
    coords: (N, 3) coordinates of all points
    num_nodes_per_dim: number of knots per dimension
    """
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    axes = [np.linspace(mins[i], maxs[i], num_nodes_per_dim) for i in range(3)]
    grid = np.meshgrid(*axes, indexing='ij')
    knots = np.stack([g.ravel() for g in grid], axis=1)
    return knots

def compute_basis_values(points, knots, theta_factor=1.0):
    """
    Compute Gaussian basis function values for each point relative to all knots.
    points: (N, 3)
    knots: (K, 3)
    theta_factor: scale parameter, theta = average knot spacing * theta_factor
    """
    tree = cKDTree(knots)
    distances, _ = tree.query(knots, k=2)
    avg_spacing = np.mean(distances[:, 1])
    theta = avg_spacing * theta_factor

    dist = distance.cdist(points, knots, metric='euclidean')
    basis = np.exp(- (dist ** 2) / (2 * theta ** 2))
    return basis

In [ ]:
# Feature filtering
def filter_features(X_train,
                    nonzero_thresh=0.01,
                    var_thresh=1e-6,
                    corr_thresh=0.98,
                    sample_ratio=0.1,
                    eps=1e-12):
    """
    Perform feature filtering based on the training set: low non-zero ratio, low variance, high correlation.
    Returns a boolean mask indicating which features to keep.
    """
    n_features = X_train.shape[1]
    mask = np.ones(n_features, dtype=bool)

    # Non-zero ratio filtering
    nonzero_ratio = np.mean(np.abs(X_train) > eps, axis=0)
    mask &= (nonzero_ratio >= nonzero_thresh)
    print(f"Non-zero ratio filtering: features remaining = {mask.sum()}")

    # Variance filtering
    variances = np.var(X_train, axis=0)
    mask &= (variances >= var_thresh)
    print(f"Variance filtering: features remaining = {mask.sum()}")

    # High correlation filtering
    remain_idx = np.where(mask)[0]
    if len(remain_idx) == 0:
        print("Warning: all features filtered, check thresholds!")
        return mask

    n_samples = X_train.shape[0]
    sample_size = min(int(n_samples * sample_ratio), 10000)
    if sample_size < n_samples:
        np.random.seed(42)
        sample_idx = np.random.choice(n_samples, size=sample_size, replace=False)
        X_sample = X_train[sample_idx][:, remain_idx]
    else:
        X_sample = X_train[:, remain_idx]

    corr_matrix = np.corrcoef(X_sample, rowvar=False)
    corr_matrix = np.nan_to_num(corr_matrix)

    n_remain = len(remain_idx)
    keep = np.ones(n_remain, dtype=bool)
    for i in range(n_remain):
        if not keep[i]:
            continue
        for j in range(i + 1, n_remain):
            if not keep[j]:
                continue
            if abs(corr_matrix[i, j]) > corr_thresh:
                keep[j] = False

    final_mask = mask.copy()
    final_mask[remain_idx[~keep]] = False
    print(f"High correlation filtering: features remaining = {final_mask.sum()}")
    return final_mask

In [ ]:
# Load data
data = pd.read_csv(r"mapped_s1_data.dat")

# Extract coordinates and raw features (all points, used for basis generation)
coordinates = data[["X", "Y", "Z"]].values
features_raw = data[["den", "sus", "res"]].values
labels = data["YXML50"].values
ZK = data["T"].values

In [ ]:
# Generate multi-resolution spatial basis functions
num_nodes_per_dim_list = [3, 5, 7]  # adjustable
knots_list = []
basis_list = []
for n in num_nodes_per_dim_list:
    knots = generate_knots(coordinates, num_nodes_per_dim=n)
    knots_list.append(knots)
    basis = compute_basis_values(coordinates, knots, theta_factor=1.2)
    basis_list.append(basis)

# Save knot grid for prediction
joblib.dump(knots_list, 'knots_list.pkl')

In [ ]:
# Fuse raw features with basis functions
basis_multi = np.concatenate(basis_list, axis=1)  # (N, 27+125+343)
merged_raw = np.concatenate([features_raw, basis_multi], axis=1)  # (N, 3+495)

# Split training and test sets
mask = (data['T'] == 1) & (data['YXML50'].isin([0, 1, 2]))
original_indices = np.where(mask)[0]

train_indices, test_indices = train_test_split(
    original_indices, test_size=0.2, random_state=42,
    stratify=labels[original_indices]
)

In [ ]:
# Feature filtering
X_train_raw = merged_raw[train_indices]
feature_mask = filter_features(X_train_raw)
merged_raw = merged_raw[:, feature_mask]
joblib.dump(feature_mask, 'feature_mask.pkl')

In [ ]:
# Standardization
scaler = StandardScaler()
scaler.fit(merged_raw[train_indices])
merged_scaled = scaler.transform(merged_raw)  # (N, final number of features)
joblib.dump(scaler, 'scaler_merged.pkl')

# Prepare features and labels
X = merged_scaled
y = labels

# Split training/test sets (using previously saved indices)
X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

In [ ]:
# Class weights
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
print(f"Automatically computed class weights: {class_weight_dict}")

# Grid search parameters
weight_variants = [None, 'balanced', class_weight_dict]
param_grid = {
    'n_estimators': [200],
    'max_depth': [15],
    'min_samples_split':  [12],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': weight_variants
}

rf = RandomForestClassifier(n_jobs=-1, random_state=42, oob_score=False)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Train model
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

# Output best parameters and cross-validation score
print("\nBest parameters:", grid_search.best_params_)
print("Best cross-validation score (macro F1):", grid_search.best_score_)

# Evaluation on test set
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)

print("\n=== Test set detailed evaluation metrics ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=[f'Class {i}' for i in classes]))

conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix:")
print(conf_matrix)

In [ ]:
# Compute additional metrics
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average='macro'),
    "recall_macro": recall_score(y_test, y_pred, average='macro'),
    "f1_macro": f1_score(y_test, y_pred, average='macro')
}
print("\nEvaluation metrics dictionary:")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

# Save model
joblib.dump(best_rf, 'best_rf_model.pkl')

import pandas as pd
import numpy as np
import joblib
from scipy.spatial import cKDTree, distance

In [ ]:
# Load objects saved during training
knots_list = joblib.load('knots_list.pkl')          # multi-resolution knot list
feature_mask = joblib.load('feature_mask.pkl')      # feature mask
scaler = joblib.load('scaler_merged.pkl')            # standardizer
model = joblib.load('best_rf_model.pkl')             # trained random forest model

data_pred = pd.read_csv(r"s50_modify_data.dat")
coords_pred = data_pred[["X", "Y", "Z"]].values
features_pred_raw = data_pred[["den", "sus", "res"]].values

# Compute multi-resolution basis functions
basis_pred_list = []
for knots in knots_list:
    basis = compute_basis_values(coords_pred, knots, theta_factor=1.2)
    basis_pred_list.append(basis)
basis_pred_multi = np.concatenate(basis_pred_list, axis=1)

# Concatenate raw features and basis functions
merged_pred_raw = np.concatenate([features_pred_raw, basis_pred_multi], axis=1)

# Apply feature mask (keep only features selected during training)
merged_pred_raw = merged_pred_raw[:, feature_mask]

# Standardize
merged_pred_scaled = scaler.transform(merged_pred_raw)

# Prediction
y_pred = model.predict(merged_pred_scaled)                 # class labels
y_proba = model.predict_proba(merged_pred_scaled)          # class probabilities

# Build output DataFrame
# Copy original data to avoid modifying the original DataFrame
output_df = data_pred.copy()
# Remove possible existing prediction columns
for col in output_df.columns:
    if col.startswith('class_') or col == "YXML50":
        output_df.drop(columns=[col], inplace=True, errors='ignore')

# Add prediction results
output_df['predicted_class'] = y_pred

num_classes = len(model.classes_)
prob_cols = [f'class_{c}' for c in model.classes_]
prob_df = pd.DataFrame(y_proba, columns=prob_cols)
output_df = pd.concat([output_df, prob_df], axis=1)

# Save results
file_path = 'prediction_random_forest.csv'
output_df.to_csv(file_path, index=False, encoding='utf-8-sig')
print(f"Results saved to {file_path}")